# Домашняя работа №5

**Игнатенко Денис Беньяминович**  
**Группа:** НПИбд-02-25  
**Студенческий билет:** 1032252476

## Добавлено:
- однородные ОДУ;
- четыре типа симметрий;
- симметрия $\xi=f(x)g(y),\;\eta=0$;
- интегрирующий множитель и первый интеграл.

In [ ]:
var('x y')

class df:
    def __init__(self, p, q):
        self.p = SR(p)
        self.q = SR(q)

    def list(self):
        return [self.p, self.q]

    def zero(self, expr):
        return bool(SR(expr).simplify_full() == 0)

    def rhs(self):
        if self.zero(self.q):
            raise ValueError("Коэффициент при dy равен нулю.")
        return (-self.p/self.q).simplify_full()

    def exact(self):
        return self.zero(diff(self.p, y) - diff(self.q, x))

    def homogeneous(self):
        k = SR.var('k')
        F = self.rhs()
        scaled = F.subs({x:k*x, y:k*y}).simplify_full()
        return self.zero(diff(scaled, k))

    def solve_homogeneous(self):
        if not self.homogeneous():
            raise ValueError("Уравнение не однородное.")
        v = SR.var('v')
        phi = self.rhs().subs({x:1, y:v}).simplify_full()
        I = integral(1/(phi-v), v)
        return (I.subs(v=y/x) - log(x)).simplify_full()

    def lie(self, xi, eta):
        F = self.rhs()
        L = (
            diff(eta, x)
            + (diff(eta, y)-diff(xi, x))*F
            - diff(xi, y)*F**2
            - xi*diff(F, x)
            - eta*diff(F, y)
        )
        return L.simplify_full()

    def symmetry_fx(self):
        F = self.rhs()
        A = (-diff(F, x)/F).simplify_full()
        if not self.zero(diff(A, y)):
            return None
        xi = exp(integral(A, x)).simplify_full()
        return (xi, 0) if self.zero(self.lie(xi, 0)) else None

    def symmetry_fy(self):
        F = self.rhs()
        B = (-diff(F, x)/F**2).simplify_full()
        if not self.zero(diff(B, x)):
            return None
        xi = exp(integral(B, y)).simplify_full()
        return (xi, 0) if self.zero(self.lie(xi, 0)) else None

    def symmetry_eta_fx(self):
        F = self.rhs()
        A = diff(F, y).simplify_full()
        if not self.zero(diff(A, y)):
            return None
        eta = exp(integral(A, x)).simplify_full()
        return (0, eta) if self.zero(self.lie(0, eta)) else None

    def symmetry_eta_fy(self):
        F = self.rhs()
        B = (diff(F, y)/F).simplify_full()
        if not self.zero(diff(B, x)):
            return None
        eta = exp(integral(B, y)).simplify_full()
        return (0, eta) if self.zero(self.lie(0, eta)) else None

    def find_product_symmetry(self):
        F = self.rhs()
        R = (diff(diff(log(F), y), x)/F**2).simplify_full()
        if self.zero(R):
            return None

        A = (diff(R, x)/R).simplify_full()
        if not self.zero(diff(A, y)):
            return None
        f = exp(integral(A, x)).simplify_full()

        B = (f*diff(1/(f*F), x)).simplify_full()
        if not self.zero(diff(B, x)):
            return None
        g = exp(integral(B, y)).simplify_full()

        xi = (f*g).simplify_full()
        if not self.zero(self.lie(xi, 0)):
            return None
        return xi

    def integrate_with_product_symmetry(self):
        xi = self.find_product_symmetry()
        if xi is None:
            raise ValueError("Симметрия не найдена.")

        mu = (1/(self.p*xi)).simplify_full()
        M = (mu*self.p).simplify_full()
        N = (mu*self.q).simplify_full()

        if not self.zero(diff(M, y)-diff(N, x)):
            raise ArithmeticError("Интегрирующий множитель не прошёл проверку.")

        U0 = integral(M, x)
        h_y = (N-diff(U0, y)).simplify_full()
        U = (U0 + integral(h_y, y)).simplify_full()
        return xi, mu, U

## 1. Однородное уравнение

\[
(x^2+y^2)\,dx+xy\,dy=0.
\]

In [ ]:
eq_h = df(x**2 + y**2, x*y)
assert eq_h.homogeneous()

I_h = eq_h.solve_homogeneous()
assert eq_h.zero(diff(I_h, x) + diff(I_h, y)*eq_h.rhs())
show(I_h == SR.var('C'))

## 2. Симметрия $\xi=f(x),\;\eta=0$

Условие для правой части ОДУ:

$$
-\frac{\Phi_x}{\Phi}=A(x)
$$

In [ ]:
eq_2 = df(-x*y, 1)
sym_2 = eq_2.symmetry_fx()
assert sym_2 is not None
xi_2, eta_2 = sym_2
assert eq_2.zero(eq_2.lie(xi_2, eta_2))
mu_2 = (1/(eq_2.p*xi_2 + eq_2.q*eta_2)).simplify_full()
exact_2 = ((eq_2.p*mu_2).diff(y) - (eq_2.q*mu_2).diff(x)).simplify_full()
assert eq_2.zero(exact_2)
show((xi_2, eta_2))
show(mu_2)
show(exact_2.numerator())

## 3. Симметрия $\xi=f(y),\;\eta=0$

Условие для правой части ОДУ:

$$
-\frac{\Phi_x}{\Phi^2}=B(y)
$$

In [ ]:
eq_3 = df(-1/(x*y + y**2 + 1), 1)
sym_3 = eq_3.symmetry_fy()
assert sym_3 is not None
xi_3, eta_3 = sym_3
assert eq_3.zero(eq_3.lie(xi_3, eta_3))
mu_3 = (1/(eq_3.p*xi_3 + eq_3.q*eta_3)).simplify_full()
exact_3 = ((eq_3.p*mu_3).diff(y) - (eq_3.q*mu_3).diff(x)).simplify_full()
assert eq_3.zero(exact_3)
show((xi_3, eta_3))
show(mu_3)
show(exact_3.numerator())

## 4. Симметрия $\xi=0,\;\eta=f(x)$

Условие для правой части ОДУ:

$$
\Phi_y=A(x)
$$

In [ ]:
eq_4 = df(-(x*y + x**2), 1)
sym_4 = eq_4.symmetry_eta_fx()
assert sym_4 is not None
xi_4, eta_4 = sym_4
assert eq_4.zero(eq_4.lie(xi_4, eta_4))
mu_4 = (1/(eq_4.p*xi_4 + eq_4.q*eta_4)).simplify_full()
exact_4 = ((eq_4.p*mu_4).diff(y) - (eq_4.q*mu_4).diff(x)).simplify_full()
assert eq_4.zero(exact_4)
show((xi_4, eta_4))
show(mu_4)
show(exact_4.numerator())

## 5. Симметрия $\xi=0,\;\eta=f(y)$

Условие для правой части ОДУ:

$$
\frac{\Phi_y}{\Phi}=B(y)
$$

In [ ]:
eq_5 = df(-x**2*exp(y), 1)
sym_5 = eq_5.symmetry_eta_fy()
assert sym_5 is not None
xi_5, eta_5 = sym_5
assert eq_5.zero(eq_5.lie(xi_5, eta_5))
mu_5 = (1/(eq_5.p*xi_5 + eq_5.q*eta_5)).simplify_full()
exact_5 = ((eq_5.p*mu_5).diff(y) - (eq_5.q*mu_5).diff(x)).simplify_full()
assert eq_5.zero(exact_5)
show((xi_5, eta_5))
show(mu_5)
show(exact_5.numerator())

## 6. Симметрия $\xi=f(x)g(y),\;\eta=0$

## Уравнение (22) из статьи

\[
y'=-\frac{(y+b)^2}{(x+a)\left(1+(y+b)^2(x+a)\sin y\right)}.
\]

In [ ]:
a, b = var('a b')
cheb = df(
    (y+b)**2,
    (x+a)*(1 + (y+b)**2*(x+a)*sin(y))
)

xi, mu, U = cheb.integrate_with_product_symmetry()

assert cheb.zero(cheb.lie(xi, 0))
assert cheb.zero(diff(mu*cheb.p, y) - diff(mu*cheb.q, x))
assert cheb.zero(diff(U, x) + diff(U, y)*cheb.rhs())

show(xi)
show(mu)
show(U == SR.var('C'))